In [1]:
from os.path import join

import pandas as pd
import load_TU_data
from otp_client import get_all_routes_for_mode, load_all_candidates
from find_similar_trip import find_similar_trip
from otp_utils import has_invalid_route_name, resolve_route_short_names
import importlib

In [2]:
print("Loading TU data...")
tu_session, tu_tur, tu_deltur, tu_stations = load_TU_data.load_tu(
    data_dir="/home/simpal/O/TU_Rejseplan/Data/TU/",
    session_file="tu_session_secret_2015_2025.xlsx",
    tur_file="tu_tur_secret_2015_2025.xlsx",
    deltur_file="tu_deltur_2015_2025.xlsx",
    stations_file = "Stationer_tudatabase.xlsx"
)
print("TU data loaded")

Loading TU data...


/home/simpal/miniconda3/envs/otp/lib/python3.13/site-packages/pandas/core/arrays/timedeltas.py:1163: RuntimeWarning: invalid value encountered in cast
  int_data = data.astype(np.int64)


TU data loaded


In [4]:
#Configurartion
mode_map = {
    #TU: OTP
    31: "BUS",
    32: "S_TRAIN",
    33: "RAIL",
    34: "SUBWAY",
    37: "TRAM",
    41: "FERRY",
    35: "BUS"
}
otp_url = "http://localhost:8080/otp/gtfs/v1"
search_window = "PT30M"

print(f"Search window: {search_window}")

Search window: PT30M


In [5]:
# RAIL, TRAM and SUBWAY are missing route name in TU.
# So taking all routes for these modes. Which will be used when modes
# that do include route name in TU only can access those routes, but for
# those that do not, all routes will be used.
otp_mode_routes_cache = {mode: get_all_routes_for_mode(mode) for mode in ["RAIL", "TRAM", "SUBWAY", "FERRY"]}

In [52]:
otp_mode_routes_cache

{'RAIL': ['510R',
  '920R',
  'RX',
  '030',
  '802',
  '031',
  '93',
  '920E',
  'IL',
  '806',
  '805',
  'IC',
  '930R',
  'RX',
  '803',
  '76',
  '710R',
  'RE76',
  '92',
  '005',
  '030',
  'EC',
  '940R',
  '804',
  'RE75',
  '75',
  '69',
  '93Tog',
  '210R',
  'RE',
  '950R',
  '005',
  'RE69',
  '031',
  '110R',
  '910',
  '410',
  '960R',
  'ICL',
  '92Tog'],
 'TRAM': ['L', 'L1', 'L2'],
 'SUBWAY': ['M4', 'M2', 'M3', 'M1'],
 'FERRY': ['Søby - Fynshav',
  'M/F Højestene',
  '992',
  'Ærøfærgerne',
  'Færge',
  '991',
  'ÆrøXpressen',
  'Samsø Rederi',
  'Hjortøboen',
  '993',
  'Læsø']}

In [6]:
tu_tur = tu_tur[tu_tur["PtPrimMode"].isin([31, 32, 33, 34, 37, 41])]
tu_tur = tu_tur[(tu_tur["DiaryYear"] == 2024)]
tu_deltur = tu_deltur[tu_deltur["TurId"].isin(tu_tur["TurId"])]

In [7]:
tu_deltur

,SessionId,TurId,Delturnr,StageMode,ModeGroup,StageDrivPass,StageLength,StageWaitMin,StageStartMsm,StageDurationMin,Route,FromStation,ToStation,n_deltur
299498,504410,2644643,1,1,1,NaN,1.0,NaN,615.0,10.0,NaN,NaN,NaN,3
299499,504410,2644643,2,31,120,2.0,15.0,5.0,630.0,35.0,500,NaN,NaN,3
299500,504410,2644643,3,1,1,NaN,0.2,NaN,665.0,2.0,NaN,NaN,NaN,3
299501,504410,2644644,1,1,1,NaN,0.2,NaN,720.0,5.0,NaN,NaN,NaN,3
299502,504410,2644644,2,31,120,2.0,15.0,9.0,734.0,35.0,500,NaN,NaN,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340437,533137,2733066,2,31,120,2.0,4.0,4.0,1057.0,14.0,1A,NaN,NaN,3
340438,533137,2733066,3,1,1,NaN,0.4,NaN,1071.0,6.0,NaN,NaN,NaN,3
340439,533137,2733067,1,1,1,NaN,0.7,NaN,1170.0,10.0,NaN,NaN,NaN,3
340440,533137,2733067,2,31,120,2.0,4.0,5.0,1185.0,13.0,1A,NaN,NaN,3


In [8]:
tu_deltur["otp_mode"] = tu_deltur["StageMode"].map(mode_map)
i_TurId = 2644737
tu_tur_row = tu_tur[tu_tur["TurId"] == i_TurId].iloc[0]
tu_deltur_sub = tu_deltur.loc[tu_deltur["TurId"] == i_TurId]

In [12]:
tu_tur_row

TurId                               2644737
SessionId                            504452
TurNr                                     1
TripCount                               1.0
DepartHH                                7.0
                            ...            
date_str                         2024-01-04
depart_dt         2024-01-04 07:25:00+01:00
depart_dt_str      2024-01-04T07:25:00+0100
arrival_dt        2024-01-04 08:59:00+01:00
arrival_dt_str     2024-01-04T08:59:00+0100
Name: 260419, Length: 64, dtype: object

In [10]:
tu_deltur_sub

,SessionId,TurId,Delturnr,StageMode,ModeGroup,StageDrivPass,StageLength,StageWaitMin,StageStartMsm,StageDurationMin,Route,FromStation,ToStation,n_deltur,otp_mode
299593,504452,2644737,1,1,1,NaN,0.5,NaN,445.0,2.0,NaN,NaN,NaN,5,NaN
299594,504452,2644737,2,31,120,2.0,2.0,3.0,450.0,7.0,5C,NaN,NaN,5,BUS
299595,504452,2644737,3,34,110,NaN,6.4,2.0,459.0,15.0,NaN,Nørrebro,København H,5,SUBWAY
299596,504452,2644737,4,33,110,NaN,28.3,20.0,494.0,30.0,NaN,København H,Trekroner,5,RAIL
299597,504452,2644737,5,1,1,NaN,1.2,NaN,524.0,15.0,NaN,NaN,NaN,5,NaN


In [13]:
tu_tur_row[["orig_lat","orig_lon","tiladrlat","tiladrlon"]]

orig_lat     55.706107
orig_lon     12.522198
tiladrlat    55.651686
tiladrlon    12.136666
Name: 260419, dtype: object

In [14]:
tu_deltur_sub

,SessionId,TurId,Delturnr,StageMode,ModeGroup,StageDrivPass,StageLength,StageWaitMin,StageStartMsm,StageDurationMin,Route,FromStation,ToStation,n_deltur,otp_mode
299593,504452,2644737,1,1,1,NaN,0.5,NaN,445.0,2.0,NaN,NaN,NaN,5,NaN
299594,504452,2644737,2,31,120,2.0,2.0,3.0,450.0,7.0,5C,NaN,NaN,5,BUS
299595,504452,2644737,3,34,110,NaN,6.4,2.0,459.0,15.0,NaN,Nørrebro,København H,5,SUBWAY
299596,504452,2644737,4,33,110,NaN,28.3,20.0,494.0,30.0,NaN,København H,Trekroner,5,RAIL
299597,504452,2644737,5,1,1,NaN,1.2,NaN,524.0,15.0,NaN,NaN,NaN,5,NaN


In [15]:
import tu_gtfs_stations_match
importlib.reload(tu_gtfs_stations_match)
from tu_gtfs_stations_match import match_tu_gtfs_stations
tu_gtfs_match_df = match_tu_gtfs_stations(
    tu_stations,
    bbox_buffer_m=1000,
    period=(19725, 20086),
    name_match_threshold = 0.6
)

In [16]:
tu_gtfs_match_df

,tu_station_id,tu_station_name,gtfs_station_id,gtfs_station_name,otp_mode,name_similarity,distance_degree
0,0,Aksel Møllers Have,1:20240102_000008603342,Aksel Møllers Have St. (Metro),SUBWAY,100.0,0.000078
1,1,Albertslund,1:20240102_000008600621,Albertslund St.,S_TRAIN,100.0,0.000339
2,2,Alken,1:20240102_000008600260,Alken St.,RAIL,100.0,0.000659
3,3,Allerød,1:20240102_000008600681,Allerød St.,S_TRAIN,100.0,0.000903
4,4,Amager Strand,1:20240102_000008603324,Amager Strand St. (Metro),SUBWAY,100.0,0.000261
...,...,...,...,...,...,...,...
509,541,Ålsgårde,1:20240102_000008601633,Ålsgårde St.,RAIL,100.0,0.000314
510,542,Åmarken,1:20240102_000008600763,Åmarken St.,S_TRAIN,100.0,0.000324
511,543,Århus H,1:20240102_000008600053,Aarhus H,RAIL,100.0,0.000333
512,544,Årslev,1:20240102_000008600561,Årslev St.,RAIL,100.0,0.000075


In [37]:
rows = []
for _, row in tu_deltur_sub.loc[tu_deltur_sub["StageMode"].isin([32, 33, 34, 37])].iterrows():
    rows.append({"otp_mode": row["otp_mode"], "tu_station_name": row["FromStation"]})
    rows.append({"otp_mode": row["otp_mode"], "tu_station_name": row["ToStation"]})

via_stopid = (
    pd.DataFrame(rows)
    .dropna(subset=["tu_station_name"])
    .drop_duplicates(subset=["otp_mode", "tu_station_name"], keep="first")
    .reset_index(drop=True)
)
via_stopid

,otp_mode,tu_station_name
0,SUBWAY,Nørrebro
1,SUBWAY,København H
2,RAIL,København H
3,RAIL,Trekroner


In [38]:
if via_stopid.empty:
    via_stopids = None
else:
    # Merge and maintain order
    merged = (
        via_stopid
        .merge(
            tu_gtfs_match_df[["otp_mode", "tu_station_name", "gtfs_station_id"]],
            on=["otp_mode", "tu_station_name"],
            how="left" # preserve order of via_stopid
        )
    )
        # Keep only rows with non-null gtfs_station_id and remove duplicates while preserving order
    via_stopids = (
        merged[merged["gtfs_station_id"].notna()]
        .drop_duplicates(subset=["gtfs_station_id"], keep='first')
        ["gtfs_station_id"]
        .tolist()
    )
via_stopids

['1:20240102_000008603339',
 '1:20240102_000008603330',
 '1:20240102_000008600626',
 '1:20240102_000008600755']

In [57]:
import otp_utils
importlib.reload(otp_utils)
from otp_utils import resolve_route_short_names

tu_deltur_sub_print_col = ["StageMode", "StageLength", "StageWaitMin", "StageDurationMin", "Route", "FromStation", "ToStation"]
route_names, route_names_ext, modes_json, modes_list = resolve_route_short_names(
    tu_deltur_sub,
    mode_map,
    otp_mode_routes_cache
)
route_names_ext

['920R',
 '93Tog',
 'M1',
 '803',
 '960R',
 '940R',
 'RX',
 '802',
 'RE',
 '805',
 'ICL',
 '710R',
 'M2',
 '210R',
 'M3',
 '75',
 '806',
 '110R',
 '005',
 '92',
 '410',
 '510R',
 '950R',
 'IC',
 'IL',
 '76',
 '93',
 '910',
 'EC',
 '804',
 '920E',
 '030',
 'RE75',
 '69',
 '92Tog',
 '031',
 'RE69',
 '5C',
 '930R',
 'RE76',
 'M4']

In [63]:
modes_list

['BUS', 'SUBWAY', 'RAIL']

In [58]:
if not modes_json:
    print(f"No valid public transport modes found for TurId: {i_TurId}")
print(f"modes_json: {modes_json}")
if any(mode in ["BUS", "S_TRAIN"] for mode in modes_list) and not route_names:
    print(f"No valid BUS and S_TRAIN route found for TurId: {i_TurId}")
if has_invalid_route_name(route_names):
    print(f"Invalid route name: {route_names}")

print(f"route_name: {route_names}")
print(f"route_names_ext: {route_names_ext}")

modes_json: [{'mode': 'BUS'}, {'mode': 'SUBWAY'}, {'mode': 'RAIL'}]
route_name: ['5C']
route_names_ext: ['920R', '93Tog', 'M1', '803', '960R', '940R', 'RX', '802', 'RE', '805', 'ICL', '710R', 'M2', '210R', 'M3', '75', '806', '110R', '005', '92', '410', '510R', '950R', 'IC', 'IL', '76', '93', '910', 'EC', '804', '920E', '030', 'RE75', '69', '92Tog', '031', 'RE69', '5C', '930R', 'RE76', 'M4']


In [59]:
modes_list

['BUS', 'SUBWAY', 'RAIL']

In [62]:
import otp_client
importlib.reload(otp_client)
from otp_client import load_all_candidates

is_bus_s_train = any(mode in ["BUS", "S_TRAIN"] for mode in modes_list)
is_rail_tram_subway_ferry = any(mode in ["RAIL", "SUBWAY", "TRAM", "FERRY"] for mode in modes_list)

if is_bus_s_train and is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names_ext,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url)
elif is_bus_s_train:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url)
elif is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=None,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url)
otp_candidates_df

,start,end,system_notice_tag,system_notice_text,iteration_id,leg_id,mode,route_short_name,distance_km,duration_min,generalized_cost,start_time,end_time,from,to,leg_geometry,start_dt
0,2024-01-04T06:49:14+01:00,2024-01-04T07:57:37+01:00,[],[],-8,0,WALK,NaN,0.071,2,175,1704347354000,1704347460000,Origin,Hulgårds Plads (Frederikssundsvej),"[(55.70599, 12.52213), (55.706, 12.52206), (55...",2024-01-04 06:49:14+01:00
1,2024-01-04T06:49:14+01:00,2024-01-04T07:57:37+01:00,[],[],-8,1,BUS,5C,1.191,5,900,1704347460000,1704347760000,Hulgårds Plads (Frederikssundsvej),Nørrebro St. (Nørrebrogade),"[(55.70574, 12.52257), (55.70574, 12.52258), (...",2024-01-04 06:49:14+01:00
2,2024-01-04T06:49:14+01:00,2024-01-04T07:57:37+01:00,[],[],-8,2,WALK,NaN,0.093,4,322,1704347760000,1704347996000,Nørrebro St. (Nørrebrogade),Nørrebro St. (Metro),"[(55.70056, 12.53895), (55.70054, 12.53899), (...",2024-01-04 06:49:14+01:00
3,2024-01-04T06:49:14+01:00,2024-01-04T07:57:37+01:00,[],[],-8,3,SUBWAY,M3,6.488,12,1444,1704348120000,1704348840000,Nørrebro St. (Metro),København H (Metro),"[(55.70055, 12.53807), (55.69998, 12.53795), (...",2024-01-04 06:49:14+01:00
4,2024-01-04T06:49:14+01:00,2024-01-04T07:57:37+01:00,[],[],-8,4,WALK,NaN,0.218,5,497,1704348840000,1704349155000,København H (Metro),København H,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",2024-01-04 06:49:14+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,2024-01-04T07:55:37+01:00,2024-01-04T09:05:37+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,5,0,WALK,NaN,1.198,18,2133,1704351337000,1704352440000,Origin,Nørrebro St. (Metro),"[(55.70599, 12.52213), (55.70592, 12.52247), (...",2024-01-04 07:55:37+01:00
80,2024-01-04T07:55:37+01:00,2024-01-04T09:05:37+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,5,1,SUBWAY,M3,6.488,12,1320,1704352440000,1704353160000,Nørrebro St. (Metro),København H (Metro),"[(55.70055, 12.53807), (55.69998, 12.53795), (...",2024-01-04 07:55:37+01:00
81,2024-01-04T07:55:37+01:00,2024-01-04T09:05:37+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,5,2,WALK,NaN,0.218,5,497,1704353160000,1704353475000,København H (Metro),København H,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",2024-01-04 07:55:37+01:00
82,2024-01-04T07:55:37+01:00,2024-01-04T09:05:37+01:00,[outside-search-window],[This itinerary is marked as deleted by the ou...,5,3,RAIL,RE,28.424,22,2085,1704353640000,1704354960000,København H,Trekroner St.,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",2024-01-04 07:55:37+01:00
